## Data notes: sprint-format races

In 2021-22 (6 races, `qualifying_format = SPRINT_RACE`) the Sunday grid was set by the Saturday sprint result, not by qualifying. Check: only 55% of drivers started where they qualified, vs 72% in normal 2021-22 races.

From 2023 the Sunday grid always comes from a qualifying session, so those weekends are unaffected.

Treatment: keep the 6 races and flag them (`is_sprint_grid`). Robustness check later: refit without them. Exclude them from the qualifying-gap and penalty analysis.

In [25]:
pd.read_sql("""
SELECT r.qualifying_format,
    COUNT(*) AS drivers,
    SUM(rr.grid_position_number = rr.qualification_position_number) AS same
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2021 AND 2022
AND rr.grid_position_number IS NOT NULL
AND rr.qualification_position_number IS NOT NULL
GROUP BY r.qualifying_format
""", con)

,qualifying_format,drivers,same
0,KNOCKOUT,733,527
1,SPRINT_RACE,112,62


In [23]:
pd.read_sql("""
SELECT year, qualifying_format, sprint_qualifying_format,
       COUNT(*) AS races
FROM race
WHERE year BETWEEN 2006 AND 2026
GROUP BY year, qualifying_format, sprint_qualifying_format
ORDER BY year
""", con)

,year,qualifying_format,sprint_qualifying_format,races
0,2006,KNOCKOUT,NaN,18
1,2007,KNOCKOUT,NaN,17
2,2008,KNOCKOUT,NaN,18
3,2009,KNOCKOUT,NaN,17
4,2010,KNOCKOUT,NaN,19
5,2011,KNOCKOUT,NaN,19
6,2012,KNOCKOUT,NaN,20
7,2013,KNOCKOUT,NaN,19
8,2014,KNOCKOUT,NaN,19
9,2015,KNOCKOUT,NaN,19


In [22]:
pd.read_sql("""
SELECT year, round, official_name, qualifying_format, sprint_qualifying_format
FROM race
WHERE id = 1049
""", con)

,year,round,official_name,qualifying_format,sprint_qualifying_format
0,2021,14,Formula 1 Heineken Gran Premio d'Italia 2021,SPRINT_RACE,None


In [20]:
no_grid[no_grid["position_text"] == "DNF"]

,race_id,year,round,date,circuit_id,driver_id,constructor_id,qualification_position_number,qualification_position_text,grid_position_number,grid_position_text,position_number,position_text
6393,1049,2021,14,2021-09-12,monza,pierre-gasly,alphatauri,NaN,DNF,NaN,NaN,NaN,DNF


In [19]:
no_grid["position_text"].value_counts()

position_text
DNS    16
DNQ     2
DNP     2
DNF     1
Name: count, dtype: int64

In [18]:
no_grid = df[df["grid_position_text"].isna()]
no_grid["position_number"].notna().value_counts()

position_number
False    21
Name: count, dtype: int64

In [17]:
from pathlib import Path

query = Path("../sql/01_race_results.sql").read_text()
df = pd.read_sql(query, con)
df.shape

(8724, 13)

## Data notes: grid positions

Blank `grid_position_number` values (2006-2025, 191 rows):

| Grid label | Meaning | Treatment |
|---|---|---|
| PL | Pit-lane start | Ranked behind the last gridded car, ordered by qualifying position |
| NaN | No grid information | Check whether the driver finished; drop only if they never started |

Missing qualifying positions (NC, EX, DSQ) don't affect the main model, which uses the grid slot. They matter later for the qualifying-gap and penalty analysis.

In [12]:
query = """
SELECT rr.grid_position_text, rr.qualification_position_text,
    COUNT(*) AS n
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2025
AND (rr.grid_position_number IS NULL
    OR rr.qualification_position_number IS NULL)
GROUP BY rr.grid_position_text, rr.qualification_position_text
ORDER BY n DESC
"""

pd.read_sql(query, con)

,grid_position_text,qualification_position_text,n
0,20,NC,26
1,PL,19,26
2,PL,20,24
3,PL,18,18
4,PL,NC,15
5,19,NC,14
6,22,NC,10
7,PL,14,9
8,PL,16,9
9,PL,10,8


In [9]:
query = """
SELECT r.year,
    COUNT(DISTINCT r.id) AS races,
    COUNT(*) AS rows,
    COUNT(*) - COUNT(rr.grid_position_number) AS no_grid,
    COUNT(*) - COUNT(rr.qualification_position_number) AS no_qual
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2026
GROUP BY r.year
ORDER BY r.year
"""

pd.read_sql(query, con)

,year,races,rows,no_grid,no_qual
0,2006,18,396,5,2
1,2007,17,374,4,0
2,2008,18,368,11,1
3,2009,17,340,9,3
4,2010,19,456,9,1
5,2011,19,454,2,14
6,2012,20,480,6,10
7,2013,19,418,1,5
8,2014,19,407,14,12
9,2015,19,380,7,12


## Data notes: non-finishers

Every non-finish has a blank `position_number`, with the reason in `position_text` (2006-2025 counts):

| Label | Count | Treatment |
|---|---|---|
| DNF | 1,355 | Counts as 0 in team form |
| NC | 8 | Counts as 0 in team form |
| DNS | 49 | Row dropped (never raced) |
| DSQ | 28 | Row dropped (rules breach, not pace) |
| DNQ / DNP | 2 / 2 | Row dropped (never raced) |

Caveat: DNF mixes crashes and mechanical failures, so team form partly reflects reliability, not just speed.

In [7]:
query = """
SELECT rr.position_text, COUNT(*) AS n
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2025
AND rr.position_number IS NULL
GROUP BY rr.position_text
ORDER BY n DESC
"""

pd.read_sql(query, con)

,position_text,n
0,DNF,1355
1,DNS,49
2,DSQ,28
3,NC,8
4,DNQ,2
5,DNP,2


In [5]:
query = """
SELECT r.year, r.round, rr.driver_id,
       rr.qualification_position_number AS qual,
       rr.grid_position_number AS grid,
       rr.position_number, rr.position_text, rr.reason_retired
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year = 2024
AND (rr.position_number IS NULL
    OR rr.qualification_position_number != rr.grid_position_number)
ORDER BY r.round
LIMIT 15
"""

pd.read_sql(query, con)

,year,round,driver_id,qual,grid,position_number,position_text,reason_retired
0,2024,2,lance-stroll,10,10,NaN,DNF,Accident
1,2024,2,pierre-gasly,18,18,NaN,DNF,Gearbox
2,2024,3,charles-leclerc,5,4,2.0,2,NaN
3,2024,3,lando-norris,4,3,3.0,3,NaN
4,2024,3,oscar-piastri,6,5,4.0,4,NaN
5,2024,3,sergio-perez,3,6,5.0,5,NaN
6,2024,3,lewis-hamilton,11,11,NaN,DNF,Power unit
7,2024,3,max-verstappen,1,1,NaN,DNF,Brakes
8,2024,4,guanyu-zhou,20,20,NaN,DNF,Gearbox
9,2024,4,daniel-ricciardo,11,11,NaN,DNF,Collision


In [3]:
query = """
SELECT r.year, r.round, rr.driver_id,rr.constructor_id,
       rr.qualification_position_number, rr.grid_position_number,
       rr.position_number, rr.position_text
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year = 2024 AND r.round = 1
ORDER BY rr.position_display_order
"""

pd.read_sql(query, con)

,year,round,driver_id,constructor_id,qualification_position_number,grid_position_number,position_number,position_text
0,2024,1,max-verstappen,red-bull,1,1,1,1
1,2024,1,sergio-perez,red-bull,5,5,2,2
2,2024,1,carlos-sainz-jr,ferrari,4,4,3,3
3,2024,1,charles-leclerc,ferrari,2,2,4,4
4,2024,1,george-russell,mercedes,3,3,5,5
5,2024,1,lando-norris,mclaren,7,7,6,6
6,2024,1,lewis-hamilton,mercedes,9,9,7,7
7,2024,1,oscar-piastri,mclaren,8,8,8,8
8,2024,1,fernando-alonso,aston-martin,6,6,9,9
9,2024,1,lance-stroll,aston-martin,12,12,10,10


In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/raw/f1db.db")